In [33]:
def extraer_campos_simples(game):

    campos_simples = {
        # IDENTIFICADORES BÁSICOS
        "game_id": "id",                                              # id juego (int)
        "game_name": "name",                                          # nombre del juego
        "tba": "tba",                                                 # To Be Announced
        # CAMPOS TEMPORALES 
        "released": "released",                                       # Fecha lanzamiento
        "update": "updated",                                          # Última actualización
        # MÉTRICAS DE VALORACIÓN
        "game_rating": "rating",                                       # Puntuación promedio
        "raiting_count": "rating_count",                               # Número total de valoraciones
        # MÉTRICAS DE ENGAGEMENT
        "game_added" : "added",                                         # Usuarios que añadieron el juego
        "playtime" : "playtime",                                        # Tiempo juego promedio
        "suggestions_count" : "suggestions_count",                      # Número de recomendaciones
        }

    #  Extrae campos simples
    resultado = {}
    for nombre_nuevo, nombre_api in campos_simples.items():
        resultado[nombre_nuevo] = game.get(nombre_api)

        # Campos anidados (GAMES_STATUS)
        added_by_status = game.get('added_by_status', {})
        resultado['yet'] = added_by_status.get('yet', 0)                 # Aún no jugado
        resultado['owned'] = added_by_status.get('owned', 0)             # Poseen el juego
        resultado['beaten'] = added_by_status.get('beaten', 0)           # Completado
        resultado['toplay'] = added_by_status.get('toplay', 0)           # En lista de pendientes
        resultado['dropped'] = added_by_status.get('dropped', 0)         # Abandonado
        resultado['playing'] = added_by_status.get('playing', 0)         # Jugando actualmente
    
        # Campos anidados (ESRB RATING)
        esrb = game.get('esrb_rating')
        if esrb:
            resultado['esrb_id'] = esrb.get('id')                         # id de la clasificación por edad 
            resultado['esrb_name'] = esrb.get('name')                     # Nombre de la categoría: everyone (E); Everyone 10+ (E10+); Teen (T); Mature (M); Adults (A); Rating pending (RP)
        else:
            resultado['esrb_id'] = None
            resultado['esrb_name'] = None
        
    return resultado
        
        


In [34]:
def extraer_ratings_distribution(game):
    
    # RATINGS_DISTRIBUTION -> Campo complejo
    # Extrae los ratings (lista de 4 elementos)
    # Retorna: lista de diccionarios
    
    game_id = game.get('id')
    ratings_list = []
    
    for rating in game.get('ratings', []):
        ratings_list.append({
            'game_id': game_id,
            'ratingd_id': rating.get('id'),                 # id ratings (distribución valoraciones)
            'ratingd_title': rating.get('title'),           # Categorías de valoración: excepcional (5 estrellas); recommended; meh; skip
            'ratingd_count': rating.get('count'),           # Número de valoraciones
            'ratingd_percent': rating.get('percent')        # Porcentajes 
        })
    
    return ratings_list


In [35]:
def extraer_platforms(game):

    # PLATFORMS -> Campo complejo
    # Extrae las plataformas (lista de 7+ elementos)
    # Retorna: lista de diccionarios
    
    game_id = game.get('id')
    platforms_list = []
    
    for platform_data in game.get('platforms', []):
        platform = platform_data.get('platform', {})
        platforms_list.append({
            'game_id': game_id,
            'platform_id': platform.get('id'),                          # id plataformas
            'platform_name': platform.get('name'),                      # Nombre de la plataforma
            'released_at': platform_data.get('released_at'),            # Lanzamiento en la plataforma
        })
    
    return platforms_list

In [36]:
# def extraer_parent_platforms(game):
#     # PARENT_PLATFORMS -> Campo complejo
#     # Extrae TODAS las parent platforms (lista de 3+ elementos)
#     # Retorna: lista de diccionarios
    
#     game_id = game.get('id')
#     parent_platforms_list = []
    
#     for pp_data in game.get('parent_platforms', []):
#         platform = pp_data.get('platform', {})
#         parent_platforms_list.append({
#             'game_id': game_id,
#             'platform_id': platform.get('id'),                     # id plataformas
#             'platform_name': platform.get('name'),                 # Nombre de la plataforma     
#             'platform_slug': platform.get('slug')                  # URL amigable de la plataforma
#         })
    
#     return parent_platforms_list


In [37]:
def extraer_genres(game):
    # GENRES -> Campo complejo
    # Extrae TODOS los géneros (lista de 1-5 elementos)
    # Retorna: lista de diccionarios
    
    game_id = game.get('id')
    genres_list = []
    
    for genre in game.get('genres', []):
        genres_list.append({
            'game_id': game_id,                                      
            'genre_id': genre.get('id'),                             # id género
            'genre_name': genre.get('name'),                          # Nombre del género
            'genre_games_count': genre.get('games_count')             # Número juegos en el género?????
        })
    
    return genres_list

In [38]:
def extraer_stores(game):
    # STORES -> Campo complejo
    # Extrae TODAS las tiendas (lista de 5+ elementos)
    # Retorna: lista de diccionarios
    
    game_id = game.get('id')
    stores_list = []
    
    for store_data in game.get('stores', []):
        store = store_data.get('store', {})
        stores_list.append({
            'game_id': game_id,
            'store_id': store.get('id'),                    # id store
            'store_name': store.get('name'),                # Nombre de la tienda
        })
    
    return stores_list

In [39]:
def extraer_tags(game):
    # TAGS -> Campo complejo
    # Extrae TODOS los tags (lista de 19+ elementos)
    # Retorna: lista de diccionarios
    
    game_id = game.get('id')
    tags_list = []
    
    for tag in game.get('tags', []):
        tags_list.append({
            'game_id': game_id,
            'tag_id': tag.get('id'),                               # id tag
            'tag_name': tag.get('name'),                           # Nombre de la etiqueta
            'tag_games_count': tag.get('games_count')              # Número de juegos en la etiqueta
        })
    
    return tags_list

In [40]:
# === FUNCIÓN PRINCIPAL DE TRANSFORMACIÓN ===

def transformar_juego_completo(game):
    
    # Transforma un juego completo en todas sus estructuras
    ## game_transformed = transformar_juego_completo(game)
    
    return {
        'game': extraer_campos_simples(game),
        'ratings_distribution': extraer_ratings_distribution(game),
        'platforms': extraer_platforms(game),
       #'parent_platforms': extraer_parent_platforms(game),
        'genres': extraer_genres(game),
        'stores': extraer_stores(game),
        'tags': extraer_tags(game)
    }
    

    

In [41]:
print("PRUEBA: EJEMPLO CON UNA PÁGINA DE LA API\n") 

import requests
import json

# Configuración
API_KEY = "a8a28ae9b66b4f6783503688ca05f98e"
base_url = f"https://api.rawg.io/api/games"

params = {
        "key"       : API_KEY,
        "page_size" : 1,
         }
response = requests.get(url = base_url, params = params)

if response.status_code == 200:
    data = response.json()
    game = data['results'][0]
    
    # Transformar juego completo
    juego_transformado = transformar_juego_completo(game)

print(json.dumps(juego_transformado, indent=2, ensure_ascii=False))

PRUEBA: EJEMPLO CON UNA PÁGINA DE LA API

{
  "game": {
    "game_id": 3498,
    "yet": 556,
    "owned": 12843,
    "beaten": 6429,
    "toplay": 646,
    "dropped": 1183,
    "playing": 778,
    "esrb_id": 4,
    "esrb_name": "Mature",
    "game_name": "Grand Theft Auto V",
    "tba": false,
    "released": "2013-09-17",
    "update": "2026-01-27T02:59:03",
    "game_rating": 4.47,
    "raiting_count": null,
    "game_added": 22435,
    "playtime": 74,
    "suggestions_count": 446
  },
  "ratings_distribution": [
    {
      "game_id": 3498,
      "ratingd_id": 5,
      "ratingd_title": "exceptional",
      "ratingd_count": 4387,
      "ratingd_percent": 59.06
    },
    {
      "game_id": 3498,
      "ratingd_id": 4,
      "ratingd_title": "recommended",
      "ratingd_count": 2428,
      "ratingd_percent": 32.69
    },
    {
      "game_id": 3498,
      "ratingd_id": 3,
      "ratingd_title": "meh",
      "ratingd_count": 469,
      "ratingd_percent": 6.31
    },
    {
      "game_

In [ ]:
# === FUNCIÓN LIMPIAR DATOS ===

def limpiar_datos():
    

In [ ]:
# === FUNCIÓN TRANSFORMACIÓN COMPLETA ===

def transformar_pipeline(games_list):
    # Transforma lista de juegos en bruto